# Week 3 — E0 Small FCN + Cross-Entropy Baseline

This notebook defines and smoke-tests the lightweight three-class FCN reference model. It verifies the model interface before training; it does **not** report a trained segmentation result.

In [1]:
import torch
from torch import nn
from torch.nn import functional as F

print(f"PyTorch: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

PyTorch: 2.10.0
Device: cpu


## Model

Architecture: `ConvBlock(3,32) → MaxPool → ConvBlock(32,64) → MaxPool → ConvBlock(64,128) → 1×1 Conv → bilinear upsampling`.

There are no U-Net skip connections. The model returns raw logits for `CrossEntropyLoss`; do not apply softmax before the loss.

In [2]:
class ConvBlock(nn.Sequential):
    """Two 3×3 convolutions with batch normalization and ReLU."""

    def __init__(self, in_channels, out_channels):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )


class SmallFCN(nn.Module):
    """Small FCN that returns logits at the input spatial resolution."""

    def __init__(self, in_channels=3, num_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(in_channels, 32),
            nn.MaxPool2d(2),
            ConvBlock(32, 64),
            nn.MaxPool2d(2),
            ConvBlock(64, 128),
        )
        self.classifier = nn.Conv2d(128, num_classes, kernel_size=1)

    def forward(self, inputs):
        input_size = inputs.shape[-2:]
        logits = self.classifier(self.features(inputs))
        return F.interpolate(
            logits, size=input_size, mode="bilinear", align_corners=False
        )


def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## Forward pass and Cross-Entropy check

The synthetic mask uses class indices `0`, `1`, and `2`, matching the cleaned dataset loader.

In [3]:
torch.manual_seed(255)

model = SmallFCN(in_channels=3, num_classes=3)
images = torch.randn(8, 3, 256, 256)
masks = torch.randint(0, 3, (8, 256, 256), dtype=torch.long)

logits = model(images)
loss = nn.CrossEntropyLoss()(logits, masks)
parameter_count = count_trainable_parameters(model)

assert logits.shape == (8, 3, 256, 256)
assert torch.isfinite(logits).all()
assert torch.isfinite(loss)

print(f"images: {tuple(images.shape)} {images.dtype}")
print(f"masks:  {tuple(masks.shape)} {masks.dtype}")
print(f"logits: {tuple(logits.shape)} {logits.dtype}")
print(f"CrossEntropyLoss: {loss.item():.6f}")
print(f"Trainable parameters: {parameter_count:,}")

images: (8, 3, 256, 256) torch.float32
masks:  (8, 256, 256) torch.int64
logits: (8, 3, 256, 256) torch.float32
CrossEntropyLoss: 1.122697
Trainable parameters: 287,843


## Next experiment step

When the Oxford-IIIT Pet data and full project dependencies are available, connect this model to `src/dataset.py` using `configs/train_val_split_clean.csv`. Train only on the training split, select checkpoints using validation mIoU/per-class IoU, and do not use the held-out test split for tuning.